In [1]:
# resume ATS score feature

In [1]:
# imports

import os
import json
import re

from dotenv import load_dotenv
from google import genai

In [2]:
# load environment variables

load_dotenv(override=True)

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY not found. Please check your .env file.")

client = genai.Client(api_key=api_key)

print("Gemini API initialized successfully.")

Gemini API initialized successfully.


In [4]:
# load parsed resume data

resume_file = "parsed_resume.json"

with open(resume_file, "r", encoding="utf-8") as file:
    resume_data = json.load(file)

print("Resume data loaded successfully.")
print(json.dumps(resume_data, indent=2, ensure_ascii=False))

Resume data loaded successfully.
{
  "personal_information": {
    "full_name": "Dipal Rana",
    "current_role": null,
    "email": "dipalrana@gmail.com",
    "phone": "+91 9876543210",
    "location": "India",
    "linkedin": "linkedin.com/in/dipalrana",
    "github": "github.com/dipalrana",
    "portfolio": null,
    "other_links": []
  },
  "summary": "Data-driven Computer Science student with strong skills in Python, Machine Learning, SQL, and Data Engineering. Experienced in building ETL pipelines, predictive models, and data-driven applications. Passionate about solving real-world problems using AI and scalable systems.",
  "technical_skills": {
    "programming_languages": [
      "Python",
      "SQL"
    ],
    "frontend": [],
    "backend": [],
    "frameworks": [],
    "libraries": [
      "Pandas",
      "NumPy",
      "Scikit-learn",
      "TensorFlow"
    ],
    "databases": [
      "MySQL",
      "MongoDB",
      "PostgreSQL"
    ],
    "cloud": [
      "AWS (Basic)"
  

In [5]:
# check resume sections

print("Resume sections:\n")

for section in resume_data:
    print("-", section)

print("\nTotal sections:", len(resume_data))

Resume sections:

- personal_information
- summary
- technical_skills
- soft_skills
- education
- experience
- projects
- internships
- certifications
- achievements
- languages
- references

Total sections: 12


In [9]:
# target job role

target_job_role = "ML Engineer"

print("Target Job Role:", target_job_role)

Target Job Role: ML Engineer


In [7]:
# prepare resume text

resume_text = json.dumps(resume_data, ensure_ascii=False)

print("Resume text prepared successfully.")
print("Total characters:", len(resume_text))

Resume text prepared successfully.
Total characters: 3344


In [8]:
# extract skills from resume

technical_skills = resume_data.get("technical_skills", [])
soft_skills = resume_data.get("soft_skills", [])

print("Technical Skills:")
print(technical_skills)

print("\nSoft Skills:")
print(soft_skills)

Technical Skills:
{'programming_languages': ['Python', 'SQL'], 'frontend': [], 'backend': [], 'frameworks': [], 'libraries': ['Pandas', 'NumPy', 'Scikit-learn', 'TensorFlow'], 'databases': ['MySQL', 'MongoDB', 'PostgreSQL'], 'cloud': ['AWS (Basic)'], 'devops': ['Docker'], 'ai_ml': ['Machine Learning', 'Deep Learning', 'NLP'], 'tools': ['Git', 'GitHub', 'Jupyter'], 'operating_systems': [], 'others': ['ETL Pipelines', 'Data Processing', 'Data Warehousing']}

Soft Skills:
[]


In [10]:
# prepare technical skills for ATS analysis

technical_skill_list = []

for category, skills in technical_skills.items():
    if isinstance(skills, list):
        technical_skill_list.extend(skills)

technical_skill_list = [skill.strip() for skill in technical_skill_list if skill.strip()]

print("Technical skills found:", len(technical_skill_list))
print(technical_skill_list)

Technical skills found: 20
['Python', 'SQL', 'Pandas', 'NumPy', 'Scikit-learn', 'TensorFlow', 'MySQL', 'MongoDB', 'PostgreSQL', 'AWS (Basic)', 'Docker', 'Machine Learning', 'Deep Learning', 'NLP', 'Git', 'GitHub', 'Jupyter', 'ETL Pipelines', 'Data Processing', 'Data Warehousing']


In [11]:
# normalize technical skills

normalized_skills = []

for skill in technical_skill_list:
    skill = skill.lower().strip()
    
    if skill not in normalized_skills:
        normalized_skills.append(skill)

print("Normalized technical skills:")
print(normalized_skills)

Normalized technical skills:
['python', 'sql', 'pandas', 'numpy', 'scikit-learn', 'tensorflow', 'mysql', 'mongodb', 'postgresql', 'aws (basic)', 'docker', 'machine learning', 'deep learning', 'nlp', 'git', 'github', 'jupyter', 'etl pipelines', 'data processing', 'data warehousing']


In [19]:
# ATS analysis prompt

ats_prompt = """
You are an expert ATS resume evaluator and career advisor.

Your task is to analyze the given resume for the preferred job role.

Preferred Job Role:
""" + target_job_role + """

Resume Data:
""" + resume_text + """

Evaluate how well this resume is prepared for the preferred job role.

The ATS score must be between 0 and 100 and must be based on the actual information present in the resume.

Do not randomly generate the score.
Do not assume that a skill exists if it is not present in the resume.
Use evidence from the resume when evaluating each section.

Consider the following areas:

1. Resume completeness
2. Skills and relevant keywords
3. Target job-role matching
4. Resume content quality
5. Work experience and internships
6. Projects and project relevance
7. Education
8. Certifications and achievements
9. ATS friendliness

Return ONLY a valid JSON object.

Do not use markdown.
Do not use ```json.
Do not explain anything outside the JSON.
Do not guess information that is not present in the resume.

Use the following JSON structure:

{
  "ats_score": 0,

  "score_breakdown": {
    "resume_completeness": 0,
    "skills_and_keywords": 0,
    "job_role_match": 0,
    "content_quality": 0,
    "experience": 0,
    "projects": 0,
    "education": 0,
    "certifications_and_achievements": 0,
    "ats_friendliness": 0
  },

  "resume_completeness": {
    "percentage": 0,
    "present_sections": [],
    "missing_sections": [],
    "analysis": ""
  },

  "skills_analysis": {
    "matched_skills": [],
    "missing_skills": [],
    "relevant_keywords": [],
    "missing_keywords": [],
    "analysis": ""
  },

  "job_role_match": {
    "match_percentage": 0,
    "matching_areas": [],
    "missing_areas": [],
    "analysis": ""
  },

  "content_quality": {
    "score": 0,
    "strengths": [],
    "weaknesses": [],
    "analysis": ""
  },

  "experience_analysis": {
    "score": 0,
    "strengths": [],
    "weaknesses": [],
    "analysis": ""
  },

  "project_analysis": {
    "score": 0,
    "relevant_projects": [],
    "strengths": [],
    "weaknesses": [],
    "analysis": ""
  },

  "strengths": [],

  "weaknesses": [],

  "improvement_suggestions": [],

  "overall_feedback": ""
}

Scoring rules:

- All scores must be between 0 and 100.
- The ats_score must be consistent with the score_breakdown.
- Do not give a high score only because many resume sections are present.
- Give importance to relevant skills, keywords, job-role relevance, projects, experience, and content quality.
- For students and freshers, do not heavily penalize the absence of professional experience.
- Relevant internships and academic projects should be considered valuable experience.
- Only mark skills as matched when there is reasonable evidence in the resume.
- Missing skills and keywords should be relevant to the preferred job role.
- Evaluate whether projects demonstrate skills relevant to the preferred job role.
- Evaluate whether experience and internships contain useful responsibilities and technologies.
- Consider whether the resume content is clear, specific, and professional.
- Consider whether the resume is likely to be readable by an ATS.
- Suggestions should be practical and suitable for students and freshers.
- Do not invent information about the candidate.
- make some short not long make like use very low token and low output 

Return only the JSON object.
"""

print("ATS analysis prompt created successfully.")

ATS analysis prompt created successfully.


In [20]:
# generate ATS response

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=ats_prompt,
    config={
        "temperature": 0.2,
        "max_output_tokens": 5200,
        "response_mime_type": "application/json"
    }
)

print("ATS analysis generated successfully.")
print(response.text)

ATS analysis generated successfully.
{
  "ats_score": 70,
  "score_breakdown": {
    "resume_completeness": 70,
    "skills_and_keywords": 75,
    "job_role_match": 70,
    "content_quality": 70,
    "experience": 65,
    "projects": 80,
    "education": 90,
    "certifications_and_achievements": 40,
    "ats_friendliness": 90
  },
  "resume_completeness": {
    "percentage": 80,
    "present_sections": [
      "personal_information",
      "summary",
      "technical_skills",
      "education",
      "projects",
      "internships",
      "achievements",
      "languages"
    ],
    "missing_sections": [
      "soft_skills (empty)",
      "certifications (empty)",
      "portfolio (empty)",
      "references (empty)"
    ],
    "analysis": "The resume includes most standard sections for a student. However, some sections like 'certifications' and 'soft_skills' are present but empty, reducing overall completeness."
  },
  "skills_analysis": {
    "matched_skills": [
      "Python",
    

In [21]:
# convert ATS response into JSON

ats_result = json.loads(response.text)

print("ATS result converted to JSON successfully.")
print(json.dumps(ats_result, indent=2, ensure_ascii=False))

ATS result converted to JSON successfully.
{
  "ats_score": 70,
  "score_breakdown": {
    "resume_completeness": 70,
    "skills_and_keywords": 75,
    "job_role_match": 70,
    "content_quality": 70,
    "experience": 65,
    "projects": 80,
    "education": 90,
    "certifications_and_achievements": 40,
    "ats_friendliness": 90
  },
  "resume_completeness": {
    "percentage": 80,
    "present_sections": [
      "personal_information",
      "summary",
      "technical_skills",
      "education",
      "projects",
      "internships",
      "achievements",
      "languages"
    ],
    "missing_sections": [
      "soft_skills (empty)",
      "certifications (empty)",
      "portfolio (empty)",
      "references (empty)"
    ],
    "analysis": "The resume includes most standard sections for a student. However, some sections like 'certifications' and 'soft_skills' are present but empty, reducing overall completeness."
  },
  "skills_analysis": {
    "matched_skills": [
      "Python"

In [22]:
# check ATS score

print("ATS Score:", ats_result["ats_score"])

print("\nScore Breakdown:")

for section, score in ats_result["score_breakdown"].items():
    print(f"{section}: {score}")

ATS Score: 70

Score Breakdown:
resume_completeness: 70
skills_and_keywords: 75
job_role_match: 70
content_quality: 70
experience: 65
projects: 80
education: 90
certifications_and_achievements: 40
ats_friendliness: 90


In [23]:
# check resume completeness

completeness = ats_result["resume_completeness"]

print("Resume Completeness:", completeness["percentage"], "%")

print("\nPresent Sections:")
for section in completeness["present_sections"]:
    print("-", section)

print("\nMissing Sections:")
for section in completeness["missing_sections"]:
    print("-", section)

print("\nAnalysis:")
print(completeness["analysis"])

Resume Completeness: 80 %

Present Sections:
- personal_information
- summary
- technical_skills
- education
- projects
- internships
- achievements
- languages

Missing Sections:
- soft_skills (empty)
- certifications (empty)
- portfolio (empty)
- references (empty)

Analysis:
The resume includes most standard sections for a student. However, some sections like 'certifications' and 'soft_skills' are present but empty, reducing overall completeness.


In [24]:
# check skills and keyword analysis

skills_analysis = ats_result["skills_analysis"]

print("Matched Skills:")
for skill in skills_analysis["matched_skills"]:
    print("-", skill)

print("\nMissing Skills:")
for skill in skills_analysis["missing_skills"]:
    print("-", skill)

print("\nRelevant Keywords:")
for keyword in skills_analysis["relevant_keywords"]:
    print("-", keyword)

print("\nMissing Keywords:")
for keyword in skills_analysis["missing_keywords"]:
    print("-", keyword)

print("\nAnalysis:")
print(skills_analysis["analysis"])

Matched Skills:
- Python
- SQL
- Pandas
- NumPy
- Scikit-learn
- TensorFlow
- MySQL
- MongoDB
- PostgreSQL
- AWS (Basic)
- Docker
- Machine Learning
- Deep Learning
- NLP
- Git
- GitHub
- Jupyter
- ETL Pipelines
- Data Processing
- Data Warehousing

Missing Skills:
- MLOps tools (e.g., MLflow, Airflow, Kubernetes)
- Advanced cloud ML services (e.g., AWS SageMaker)
- Distributed computing (e.g., Spark)
- Model deployment frameworks (e.g., Flask, FastAPI)
- Software engineering best practices for ML systems

Relevant Keywords:
- Machine Learning
- Deep Learning
- NLP
- predictive models
- ETL pipelines
- data processing
- scalable systems

Missing Keywords:
- model deployment
- MLOps
- production
- scalability
- optimization
- API
- real-time

Analysis:
The resume demonstrates a strong foundation in core ML, data processing, and relevant programming languages. However, it lacks specific skills and keywords related to MLOps, model deployment, and productionizing ML systems, which are cruc

In [25]:
# check job role matching

job_role_match = ats_result["job_role_match"]

print("Job Role Match:", job_role_match["match_percentage"], "%")

print("\nMatching Areas:")
for area in job_role_match["matching_areas"]:
    print("-", area)

print("\nMissing Areas:")
for area in job_role_match["missing_areas"]:
    print("-", area)

print("\nAnalysis:")
print(job_role_match["analysis"])

Job Role Match: 70 %

Matching Areas:
- Core Machine Learning knowledge (Scikit-learn, TensorFlow, NLP)
- Strong Python and SQL skills
- Experience with data processing and ETL pipelines
- Exposure to cloud (AWS Basic) and containerization (Docker)
- Relevant academic background

Missing Areas:
- Practical experience in deploying and maintaining ML models in production
- Knowledge of MLOps tools and practices
- Advanced cloud ML services and infrastructure
- Strong software engineering focus for scalable ML systems

Analysis:
The candidate is well-suited for an entry-level ML role focused on model development and data analysis. To fully align with an ML Engineer role, more emphasis on MLOps, deployment, and scalable system design is needed.


In [26]:
# check resume content quality

content_quality = ats_result["content_quality"]

print("Content Quality Score:", content_quality["score"])

print("\nStrengths:")
for strength in content_quality["strengths"]:
    print("-", strength)

print("\nWeaknesses:")
for weakness in content_quality["weaknesses"]:
    print("-", weakness)

print("\nAnalysis:")
print(content_quality["analysis"])

Content Quality Score: 70

Strengths:
- Clear and concise summary highlighting key skills.
- Specific technical skills are well-categorized.
- Internship descriptions use action verbs and list relevant technologies.

Weaknesses:
- Project descriptions lack quantifiable results or impact.
- Achievements are general and lack specific details or metrics.
- The 'certifications' section is empty, despite an achievement mentioning them.

Analysis:
The content is clear and professional but could be significantly enhanced by adding quantifiable achievements and specific outcomes for projects and internships. The empty certifications section is a missed opportunity.


In [27]:
# check experience and internship analysis

experience_analysis = ats_result["experience_analysis"]

print("Experience Score:", experience_analysis["score"])

print("\nStrengths:")
for strength in experience_analysis["strengths"]:
    print("-", strength)

print("\nWeaknesses:")
for weakness in experience_analysis["weaknesses"]:
    print("-", weakness)

print("\nAnalysis:")
print(experience_analysis["analysis"])

Experience Score: 65

Strengths:
- Two relevant internships (Data Science, Data Engineering) provide practical experience.
- Internships demonstrate application of core ML and data processing skills (Pandas, NumPy, Scikit-learn, Python, SQL).
- Responsibilities include building ML models and developing ETL pipelines.

Weaknesses:
- Internships are short-term (2 months each).
- No full-time professional experience.
- Descriptions could benefit from quantifiable results or impact achieved during the internships.

Analysis:
For a student, the internships are valuable and directly relevant. They showcase hands-on experience in data science and data engineering, which are foundational for an ML Engineer. Quantifying achievements would make them more impactful.


In [28]:
# check project analysis

project_analysis = ats_result["project_analysis"]

print("Project Score:", project_analysis["score"])

print("\nRelevant Projects:")
for project in project_analysis["relevant_projects"]:
    print("-", project)

print("\nStrengths:")
for strength in project_analysis["strengths"]:
    print("-", strength)

print("\nWeaknesses:")
for weakness in project_analysis["weaknesses"]:
    print("-", weakness)

print("\nAnalysis:")
print(project_analysis["analysis"])

Project Score: 80

Relevant Projects:
- End-to-End Data Pipeline
- AI-Based Prediction System
- NLP Text Analyzer

Strengths:
- Projects directly showcase core ML, NLP, and data engineering skills.
- Cover different aspects of the ML lifecycle from data processing to model development.
- Demonstrate practical application of Python, SQL, Scikit-learn, and NLP techniques.

Weaknesses:
- No GitHub links provided for projects, hindering verification and deeper review.
- Project descriptions lack specific metrics, challenges, or quantifiable outcomes.
- The 'scalable data storage system' claim lacks detail.

Analysis:
The projects are highly relevant and demonstrate a good understanding of ML and data pipelines. Providing GitHub links and adding quantifiable results would significantly strengthen this section.


In [29]:
# check overall strengths, weaknesses, and suggestions

print("Overall Strengths:")
for strength in ats_result["strengths"]:
    print("-", strength)

print("\nOverall Weaknesses:")
for weakness in ats_result["weaknesses"]:
    print("-", weakness)

print("\nImprovement Suggestions:")
for suggestion in ats_result["improvement_suggestions"]:
    print("-", suggestion)

print("\nOverall Feedback:")
print(ats_result["overall_feedback"])

Overall Strengths:
- Strong academic background in Computer Science Engineering with excellent CGPA.
- Relevant internships in Data Science and Data Engineering.
- Solid foundational skills in Python, SQL, Machine Learning, Deep Learning, and NLP.
- Relevant projects demonstrating practical application of ML and data processing.
- Good understanding of ETL pipelines and data warehousing concepts.

Overall Weaknesses:
- Lack of specific MLOps, model deployment, and production-level ML engineering skills.
- Absence of quantifiable results in project and internship descriptions.
- No GitHub links for projects.
- Empty certifications section, despite mentioning certifications in achievements.
- Achievements are generic and lack specific details.

Improvement Suggestions:
- Gain practical experience or complete projects focused on MLOps, model deployment, and scaling ML systems (e.g., using MLflow, Airflow, Kubernetes, Flask/FastAPI).
- Quantify achievements in internships and projects with

In [30]:
# validate ATS score

ats_score = ats_result.get("ats_score")

if isinstance(ats_score, (int, float)) and 0 <= ats_score <= 100:
    print("ATS score is valid:", ats_score)
else:
    raise ValueError("Invalid ATS score. Score must be between 0 and 100.")

ATS score is valid: 70


In [31]:
# validate score breakdown

score_breakdown = ats_result.get("score_breakdown", {})

for section, score in score_breakdown.items():
    if not isinstance(score, (int, float)) or not 0 <= score <= 100:
        raise ValueError(
            f"Invalid score for {section}. Score must be between 0 and 100."
        )

print("All score breakdown values are valid.")

for section, score in score_breakdown.items():
    print(f"{section}: {score}")

All score breakdown values are valid.
resume_completeness: 70
skills_and_keywords: 75
job_role_match: 70
content_quality: 70
experience: 65
projects: 80
education: 90
certifications_and_achievements: 40
ats_friendliness: 90


In [32]:
# check ATS score consistency

breakdown_average = sum(score_breakdown.values()) / len(score_breakdown)

print("ATS Score:", ats_score)
print("Breakdown Average:", round(breakdown_average, 2))

difference = abs(ats_score - breakdown_average)

print("Difference:", round(difference, 2))

if difference <= 15:
    print("ATS score is reasonably consistent with the score breakdown.")
else:
    print("Warning: ATS score differs significantly from the score breakdown.")

ATS Score: 70
Breakdown Average: 72.22
Difference: 2.22
ATS score is reasonably consistent with the score breakdown.


In [33]:
# save ATS result

output_file = "ats_result.json"

with open(output_file, "w", encoding="utf-8") as file:
    json.dump(ats_result, file, indent=2, ensure_ascii=False)

print("ATS result saved successfully.")
print("File:", output_file)

ATS result saved successfully.
File: ats_result.json
